In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from statsforecast import StatsForecast
from statsforecast.models import (Naive, WindowAverage, RandomWalkWithDrift,
                                   AutoARIMA, AutoETS, SeasonalNaive, SeasonalWindowAverage)

c:\Users\pc\Desktop\Proje_2\BF-final\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
# --- Load Data ---
sales = pd.read_csv("sales_data.csv", parse_dates=["date"]).rename(columns={'date':'ds','store_id':'unique_id','sales':'y'})
future = pd.read_csv("future_values.csv", parse_dates=["date"]).rename(columns={'date':'ds','store_id':'unique_id'})
meta = pd.read_csv("metadata.csv").rename(columns={'store_id':'unique_id'})

C:\Users\pc\AppData\Local\Temp\ipykernel_19428\493605631.py:2: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  sales = pd.read_csv("sales_data.csv", parse_dates=["date"]).rename(columns={'date':'ds','store_id':'unique_id','sales':'y'})


In [10]:
# --- Metadata Encoding ---
meta_encoded = pd.get_dummies(meta, columns=['store_type', 'assortment'], drop_first=True)
for col in meta_encoded.columns:
    if meta_encoded[col].dtype == bool:
        meta_encoded[col] = meta_encoded[col].astype(int)

meta_encoded.head()

,unique_id,competition_distance,store_type_b,store_type_c,store_type_d,assortment_b,assortment_c
0,store_1,1270.0,0,1,0,0,0
1,store_2,14130.0,0,0,0,0,0
2,store_3,24000.0,0,0,0,0,1
3,store_4,7520.0,0,0,0,0,0
4,store_5,2030.0,0,0,0,0,1


In [11]:
# --- Sales Data Preparation ---
sales['state_holiday'] = sales['state_holiday'].astype(str)
sales = pd.get_dummies(sales, columns=['state_holiday'], prefix='holiday', drop_first=True)
for col in sales.columns:
    if col.startswith('holiday_') and sales[col].dtype == bool:
        sales[col] = sales[col].astype(int)

sales = sales.merge(meta_encoded, on='unique_id', how='left')
sales['ds'] = sales['ds'].dt.to_period('W').apply(lambda r: r.start_time)

agg_sales = {
    'y': 'sum',
    'customers': 'sum',
    'open': 'mean',
    'promo': 'mean',
    'school_holiday': 'mean',
    'competition_distance': 'first',
    'holiday_a': 'sum',
    'holiday_b': 'sum',
    'holiday_c': 'sum',
    **{col: 'first' for col in meta_encoded.columns if col != 'unique_id'}
}

weekly_sales = sales.groupby(['unique_id', 'ds']).agg(agg_sales).reset_index()
weekly_sales['competition_distance'].fillna(weekly_sales['competition_distance'].median(), inplace=True)
weekly_sales.head()

C:\Users\pc\AppData\Local\Temp\ipykernel_19428\3250343765.py:25: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  weekly_sales['competition_distance'].fillna(weekly_sales['competition_distance'].median(), inplace=True)


,unique_id,ds,y,customers,open,promo,school_holiday,competition_distance,holiday_a,holiday_b,holiday_c,store_type_b,store_type_c,store_type_d,assortment_b,assortment_c
0,store_1,2013-01-07,32952,3918,0.857143,0.714286,0.714286,1270.0,0,0,0,0,1,0,0,0
1,store_1,2013-01-14,25978,3417,0.857143,0.000000,0.000000,1270.0,0,0,0,0,1,0,0,0
2,store_1,2013-01-21,33071,3862,0.857143,0.714286,0.000000,1270.0,0,0,0,0,1,0,0,0
3,store_1,2013-01-28,28693,3561,0.857143,0.000000,0.000000,1270.0,0,0,0,0,1,0,0,0
4,store_1,2013-02-04,35771,4094,0.857143,0.714286,0.000000,1270.0,0,0,0,0,1,0,0,0


In [ ]:
# --- Prepare df for StatsForecast ---
#df = weekly_sales.rename(columns={'store_id': 'unique_id', 'week': 'ds', 'sales': 'y'})


# --- Future Data Preparation for Forecasting ---
future['state_holiday'] = future['state_holiday'].astype(str)
future = pd.get_dummies(future, columns=['state_holiday'], prefix='holiday', drop_first=True)
# Ensure all required holiday columns exist
for col in ['holiday_a', 'holiday_b', 'holiday_c']:
    if f'holiday_{col[-1]}' not in future.columns:
        future[f'holiday_{col[-1]}'] = 0

#for col in future.columns:
 #   if col.startswith('holiday_') and future[col].dtype == bool:
  #      future[col] = future[col].astype(int)

# One-hot encode state_holiday
#future = pd.get_dummies(future, columns=['state_holiday'], prefix='holiday', drop_first=False)

future = future.merge(meta_encoded, on='unique_id', how='left')
future['ds'] = future['ds'].dt.to_period('W').apply(lambda r: r.start_time)

agg_future = {
    'promo': 'mean',
    'school_holiday': 'mean',
    'customers': 'mean',
    'competition_distance': 'first',
    'holiday_a': 'sum',
    'holiday_b': 'sum',
    'holiday_c': 'sum',
    **{col: 'first' for col in meta_encoded.columns if col != 'unique_id'}
}

#weekly_future = future.groupby(['unique_id', 'week']).agg(agg_future).reset_index()
#X_df = weekly_future.rename(columns={'store_id': 'unique_id', 'week': 'ds'})
X_df = future.groupby(['unique_id', 'ds']).agg(agg_future).reset_index()
X_df.head()

,unique_id,ds,promo,school_holiday,customers,competition_distance,holiday_a,holiday_b,holiday_c,store_type_b,store_type_c,store_type_d,assortment_b,assortment_c
0,store_1,2015-07-20,0.000000,0.0,NaN,1270.0,0,0,0,0,1,0,0,0
1,store_1,2015-07-27,0.714286,1.0,NaN,1270.0,0,0,0,0,1,0,0,0
2,store_1,2015-08-03,0.714286,1.0,NaN,1270.0,0,0,0,0,1,0,0,0
3,store_1,2015-08-10,0.000000,1.0,NaN,1270.0,0,0,0,0,1,0,0,0
4,store_1,2015-08-17,0.714286,1.0,NaN,1270.0,0,0,0,0,1,0,0,0


In [13]:
# --- Modeling and Forecasting ---
models = [
    Naive(), 
    WindowAverage(window_size=12), 
    RandomWalkWithDrift(),
    AutoARIMA(season_length=52),
    AutoETS(season_length=52),
    SeasonalNaive(season_length=52),
    SeasonalWindowAverage(season_length=52, window_size=12)
]

sf = StatsForecast(models=models, freq='W')

In [17]:
df = weekly_sales.copy().drop(columns={'open'})

# Filter only store_1 to store_99 (inclusive)
first_100_stores = [f'store_{i}' for i in range(1, 100)]  # store_1 to store_99
df = df[df['unique_id'].isin(first_100_stores)]

df.head()

,unique_id,ds,y,customers,promo,school_holiday,competition_distance,holiday_a,holiday_b,holiday_c,store_type_b,store_type_c,store_type_d,assortment_b,assortment_c
0,store_1,2013-01-07,32952,3918,0.714286,0.714286,1270.0,0,0,0,0,1,0,0,0
1,store_1,2013-01-14,25978,3417,0.000000,0.000000,1270.0,0,0,0,0,1,0,0,0
2,store_1,2013-01-21,33071,3862,0.714286,0.000000,1270.0,0,0,0,0,1,0,0,0
3,store_1,2013-01-28,28693,3561,0.000000,0.000000,1270.0,0,0,0,0,1,0,0,0
4,store_1,2013-02-04,35771,4094,0.714286,0.000000,1270.0,0,0,0,0,1,0,0,0


In [ ]:
# --- Cross-validation ---
cv_df = sf.cross_validation(df=df, step_size=4, n_windows=5, h=8)
cv_df.to_csv("cv_forecast_results.csv", index=False)

#29 dk sürdü

In [19]:
# --- Final Forecast ---
fc_df = sf.forecast(df=df, h=8, X_df=X_df)
fc_df.to_csv("final_forecast_results.csv", index=False)

print("Forecasting pipeline complete. Forecasts saved.")

ValueError: Expected X to have shape (792, 14), but got (6084, 14)